In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import logging
import traceback
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/CAMES/data_collection_training")

#P11 and P13 removed (eye tracker did not work)
PARTICIPANTS = ["P12"] #[f"P{i:02d}" for i in range(12, 13) if i not in (11, 13)]

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
# ── Helper: parse the .txt marker file ────────────────────────────────────────
def parse_marker_file(txt_path: Path) -> pd.DataFrame:
    """
    Parse a session .txt marker file into a DataFrame.

    Columns returned:
        event_type  : 'task', 'exploration', 'feedback', '1st_measurements'
        start_s     : start time in seconds from recording start
        end_s       : end time in seconds from recording start
        label       : event label (e.g. 'task1_5_2', 'exploration')

    TXT column layout (tab-separated):
        [0] event_type
        [1] empty
        [2] start HH:MM:SS.mmm
        [3] start_s (float)
        [4] end   HH:MM:SS.mmm
        [5] end_s (float)
        [6] duration HH:MM:SS.mmm
        [7] duration_s
        [8] label
    """
    rows = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 9:
                continue
            rows.append({
                "event_type": parts[0].strip(),
                "start_s":    float(parts[3]),
                "end_s":      float(parts[5]),
                "label":      parts[8].strip(),
            })
    return pd.DataFrame(rows)

In [ ]:
# ── Helper: assign task + exploration labels to gaze samples ──────────────────
def mark_gaze(gaze: pd.DataFrame, markers: pd.DataFrame, unix_offset: float) -> pd.DataFrame:
    """
    Mark each gaze sample with task and exploration context.

    unix_offset = gaze['Timestamp Unix'].min()
    txt_seconds + unix_offset → Unix timestamp of that event
    """
    gaze = gaze.copy()

    # Initialise columns
    gaze["task_label"] = None
    gaze["task_label"] = gaze["task_label"].astype("object")
    gaze["task_begin_unix"]  = np.nan
    gaze["t_rel_to_task_s"]  = np.nan
    gaze["in_exploration"]   = False
    gaze["exploration_id"]   = np.nan

    t = gaze["Timestamp Unix"].to_numpy()

    # ── Task windows ──────────────────────────────────────────────────────────
    tasks = markers[markers["event_type"] == "task"].copy()
    tasks["start_unix"] = tasks["start_s"] + unix_offset
    tasks["end_unix"]   = tasks["end_s"]   + unix_offset

    for _, row in tasks.iterrows():
        mask = (t >= row["start_unix"]) & (t <= row["end_unix"])
        gaze.loc[mask, "task_label"]      = row["label"]
        gaze.loc[mask, "task_begin_unix"] = row["start_unix"]
        gaze.loc[mask, "t_rel_to_task_s"] = (
            gaze.loc[mask, "Timestamp Unix"] - row["start_unix"]
        )

    # ── Exploration windows ───────────────────────────────────────────────────
    explorations = markers[markers["event_type"] == "exploration"].copy()
    explorations["start_unix"] = explorations["start_s"] + unix_offset
    explorations["end_unix"]   = explorations["end_s"]   + unix_offset

    # Assign exploration_id within each task by ordering exploration windows
    # that fall inside that task chronologically
    for task_label, task_group in tasks.groupby("label", sort=False):
        # Get exploration windows that fall within this task's time range
        # (a task may appear more than once in the txt if it was done twice)
        for _, task_row in task_group.iterrows():
            exp_in_task = explorations[
                (explorations["start_unix"] >= task_row["start_unix"]) &
                (explorations["end_unix"]   <= task_row["end_unix"])
            ].sort_values("start_unix").reset_index(drop=True)

            for exp_idx, exp_row in exp_in_task.iterrows():
                mask = (t >= exp_row["start_unix"]) & (t <= exp_row["end_unix"])
                gaze.loc[mask, "in_exploration"] = True
                gaze.loc[mask, "exploration_id"] = exp_idx

    return gaze

In [ ]:
def main():
    log.info("Processing %d participant(s): %s", len(PARTICIPANTS), PARTICIPANTS)

    for pid in PARTICIPANTS:
        log.info("── %s ──────────────────────────────────────────", pid)

        # Build paths
        session_dir = BASE_DIR / pid / f"{pid}_session1"
        gaze_path   = session_dir / f"eye_{pid}" / "gaze_data.csv"
        txt_path    = session_dir / "event_markers" / f"{pid}.txt"
        out_path    = session_dir / f"eye_{pid}" / "gaze_marked.csv"

        # Check files exist
        if not gaze_path.exists():
            log.error("  Gaze file not found: %s", gaze_path)
            continue
        if not txt_path.exists():
            log.error("  Marker file not found: %s", txt_path)
            continue

        try:
            # 1. Load gaze — auto-detect separator
            with open(gaze_path) as f:
                first_line = f.readline()
            sep = ";" if ";" in first_line else ","
            log.info("  Detected separator: %r", sep)

            gaze = pd.read_csv(gaze_path, sep=sep)
            gaze["task_label"] = None
            gaze["task_label"] = gaze["task_label"].astype("object")
            gaze["Timestamp Unix"] = pd.to_numeric(gaze["Timestamp Unix"], errors="raise")
            gaze = gaze.sort_values("Timestamp Unix").reset_index(drop=True)
            log.info("  Gaze samples: %d", len(gaze))

            # 2. Compute unix offset for alignment
            unix_offset = gaze["Timestamp Unix"].min()

            # 3. Parse marker file
            markers = parse_marker_file(txt_path)
            n_tasks = (markers["event_type"] == "task").sum()
            n_exp   = (markers["event_type"] == "exploration").sum()
            log.info("  Markers: %d task windows, %d exploration windows", n_tasks, n_exp)

            # 4. Mark gaze
            gaze = mark_gaze(gaze, markers, unix_offset)

            # 5. QC report
            in_task = gaze["task_label"].notna().sum()
            in_exp  = gaze["in_exploration"].sum()
            log.info(
                "  Samples in task: %d (%.1f%%)   in exploration: %d (%.1f%%)",
                in_task, in_task / len(gaze) * 100,
                in_exp,  in_exp  / len(gaze) * 100,
            )
            log.info("  Task labels found: %s", sorted(gaze["task_label"].dropna().unique()))

            # 6. Save
            gaze.to_csv(out_path, index=False)
            log.info("  Saved → %s", out_path)

        except Exception:
            log.error("  FAILED:\n%s", traceback.format_exc())

    log.info("All done.")


main()

NOTES:

The output gaze_marked.csv is identical to the input gaze_data.csv but with five new columns added:

task_label — string like task1_5_2 or None if outside a task

task_begin_unix — Unix timestamp of when that task started

t_rel_to_task_s — seconds since task start

in_exploration — True or False

exploration_id — which exploration window (0-indexed) within the task